In [6]:
// this cell is most common to all candle notbooks in this subcrate
:toolchain stable
:dep candle-notebooks = "0.1.0"
use candle_notebooks::*;
use std::fs;
let device = Device::Cpu;
set_notebook_cwd().expect("failed to set notebook cwd");
fs::create_dir_all("images_store").ok();
println!("Toolchain: stable | Device: {:?} | CWD: {}", device, std::env::current_dir().unwrap().display());

Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
Toolchain: stable | Device: Cpu | CWD: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks


Toolchain: stable


# Tensor Math Filling Demo (Candle)

This notebook showcases generating tensors from analytical mathematical expressions using Candle.

We build coordinate grids and evaluate formulas like:
- $f(x,y) = \sin(a x) \cos(b y)$
- $g(x,y) = e^{-((x-x_0)^2 + (y-y_0)^2)/(2\sigma^2)}$ (Gaussian bump)
- Radial distance: $r(x,y) = \sqrt{(x-x_c)^2 + (y-y_c)^2}$ (normalized)
- Checkerboard: $c(x,y) = ((\lfloor x / s \rfloor + \lfloor y / s \rfloor) \bmod 2)$

We also parse free-form expressions using the helper crate’s expression module
(`ExprEnv` and `eval_expr`) so strings like `sin(6*x)*cos(4*y)` evaluate over X,Y.
See the “Parsed Expression Integration (B3)” section for examples and timings.

In [48]:

// Import required modules
use candle_notebooks::{Tensor, DType, Device};
use candle_notebooks::ah as anyhow; // central anyhow re-export
use anyhow::Result;
use std::f64::consts::PI;
use candle_notebooks as nb;

println!("✓ Dependencies loaded: candle-notebooks (anyhow via helper crate)");

// Initialize workspace
nb::set_notebook_cwd().unwrap();
nb::set_image_store_rel_dir("images_store").unwrap();
std::fs::create_dir_all("images_store").ok();

println!("✓ Notebook initialized successfully!");

✓ Dependencies loaded: candle-notebooks (anyhow via helper crate)
Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
✓ Notebook initialized successfully!


Saving images to: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks/images_store

In [12]:
// Set cwd to repo root for consistent relative paths.
candle_notebooks::set_notebook_cwd().unwrap();
println!("CWD set to repo root: {:?}", std::env::current_dir().unwrap());

// Configure image store output directory for saved PNGs.
use candle_notebooks as nb;
nb::set_image_store_rel_dir("images_store").unwrap();
println!("Image store directory: images_store");

Notebook CWD set to repository root: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks
CWD set to repo root: "/home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks"
Image store directory: images_store


Saving images to: /home/rustuser/projects/rust/from_github/candle/0aEXPLORATION/research/notebooks/candle_notebooks/images_store

## Coordinate Grid Helper
We create a function to build 2D coordinate grids X, Y over ranges with optional centering.

In [15]:
fn make_xy(nx: usize, ny: usize, x_min: f64, x_max: f64, y_min: f64, y_max: f64, device: &Device) -> candle_notebooks::CandleResult<(Tensor, Tensor)> {
    let xs: Vec<f64> = (0..nx).map(|i| x_min + (x_max - x_min) * (i as f64) / (nx as f64 - 1.0)).collect();
    let ys: Vec<f64> = (0..ny).map(|j| y_min + (y_max - y_min) * (j as f64) / (ny as f64 - 1.0)).collect();
    let x: Tensor = Tensor::new(xs, device)?.reshape((nx, 1))?;
    let y: Tensor = Tensor::new(ys, device)?.reshape((1, ny))?;
    // broadcast to (nx, ny)
    let x_var: Tensor = x.broadcast_as((nx, ny))?;
    let y_var: Tensor = y.broadcast_as((nx, ny))?;
    Ok((x_var, y_var))
}
let (nx, ny) = (128, 128);
let (x_var, y_var): (Tensor, Tensor) = make_xy(nx, ny, -1.0, 1.0, -1.0, 1.0, &device).unwrap();
println!("Grid shape: {:?}", x_var.shape());

Grid shape: [128, 128]


## Analytical Functions
We've implemented the following with Candle broadcasting:
- Sin-cos pattern: f(x,y) = sin(a x) · cos(b y)
- Gaussian bump centered at (0,0)
- Radial distance (normalized)
- Simple checkerboard approximation

Note: exp(t) computes e^t element-wise on tensors.

In [20]:
// Wrap in a closure to allow `?` and return the computed tensor `f` to notebook scope
let f: Tensor = (|| -> candle_notebooks::CandleResult<Tensor> {
    // sin-cos pattern f(x,y) = sin(a x) * cos(b y)
    let a = 6.0f64;
    let b = 4.0f64;
    let xa = (x_var.clone() * a)?;
    let yb = (y_var.clone() * b)?;
    let f = (xa.sin()? * yb.cos()?)?;
    let f_flat = f.flatten_all()?;
    let f_min = f_flat.min(0)?.to_scalar::<f64>()?;
    let f_max = f_flat.max(0)?.to_scalar::<f64>()?;
    println!("f stats -> min: {:.3} max: {:.3}", f_min, f_max);

    // Gaussian bump centered at (0,0) with sigma=0.35
    let sigma = 0.35f64;
    let x2 = (x_var.clone() * x_var.clone())?;
    let y2 = (y_var.clone() * y_var.clone())?;
    let r2 = (x2 + y2)?;
    let sigma_sq = 2.0 * sigma * sigma;
    let neg_exp_arg = (r2.clone() / sigma_sq)?;
    let g = (neg_exp_arg * -1.0)?.exp()?;
    let g_flat = g.flatten_all()?;
    let g_min = g_flat.min(0)?.to_scalar::<f64>()?;
    let g_max = g_flat.max(0)?.to_scalar::<f64>()?;
    println!("g stats -> min: {:.3} max: {:.3}", g_min, g_max);

    // Radial distance normalized (not used later, but keep for completeness)
    let r = r2.sqrt()?;
    let r_flat = r.clone().flatten_all()?;
    let r_max_val = r_flat.max(0)?.to_scalar::<f64>()?;
    let _r_norm = (r / r_max_val)?;

    // Simple checkerboard approximation (not used later)
    let s = 0.15;
    let x_scaled = (x_var.clone() / s)?.floor()?;
    let y_scaled = (y_var.clone() / s)?.floor()?;
    let _sum_scaled = (x_scaled + y_scaled)?;

    println!("Analytical functions computed successfully");
    Ok(f)
})().unwrap();

f stats -> min: -1.000 max: 1.000
g stats -> min: 0.000 max: 0.999
Analytical functions computed successfully
g stats -> min: 0.000 max: 0.999
Analytical functions computed successfully


## Normalization Helper for Image Saving
We map a tensor to 0..255 u8 for quick visualization (future: integrate with image store).

In [19]:
fn to_u8(t: &Tensor) -> candle_notebooks::CandleResult<Vec<u8>> {
    let t_flat = t.flatten_all()?;
    let min = t_flat.min(0)?.to_scalar::<f64>()?;
    let max = t_flat.max(0)?.to_scalar::<f64>()?;
    let span = max - min;
    
    // Avoid div-by-zero: if span==0 just zeros
    let norm = if span == 0.0 {
        (t - min)?
    } else {
        ((t - min)? / span)?
    };
    let scaled = (norm * 255.0)?;
    let scaled_vec = scaled.flatten_all()?.to_vec1::<f64>()?;
    Ok(scaled_vec.into_iter().map(|v| v.round().clamp(0.0, 255.0) as u8).collect())
}

(|| -> candle_notebooks::CandleResult<()> {
    let f_u8 = to_u8(&f)?;
    println!("u8 buffer length (f): {}", f_u8.len());
    Ok(())
})().unwrap();

u8 buffer length (f): 16384


## Expression Parsing (Implemented)

We parse and evaluate string expressions using `ExprEnv` and `eval_expr` from the
helper crate. This ties symbolic strings (like `sin(6*x)*cos(4*y)`) to the same
X, Y grid and parameter map used elsewhere.

Highlights:
1. `ExprEnv::new(x, y, params)` binds symbols and parameters.
2. `eval_expr("expr", &env)` parses and evaluates to a Tensor (f32).
3. Optional manual comparators quantify numerical differences.

See the section “Parsed Expression Integration (B3)” for concrete examples:
- Baselines (radial, checker, gaussian)
- Parameterized Gaussian (a, b, sigma)
- Combo expression with sin/cos + gaussian
- Timing snapshots and max_abs_diff reporting

## Next ideas and related notebooks

A lot of this is already covered in the gallery and demos. Pointers:

- Expression-driven patterns, quick timings, and image export — see [tensor_art_gallery.ipynb](tensor_art_gallery.ipynb).
- Parameter sweeps (varying a, b, sigma) and richer compositions — see the Parameter Exploration section in [tensor_art_gallery.ipynb](tensor_art_gallery.ipynb).
- Live UI exploration and sliders with an egui window — see [egui_window_demo.ipynb](egui_window_demo.ipynb).
- Image saving to `images_store/` — implemented in [tensor_art_gallery.ipynb](tensor_art_gallery.ipynb); can be ported here using `to_u8` or the helper’s image APIs.
- 3D/time stacks (introduce t and sweep/stack along a new dimension) — good follow-up; trivial with the same eval pattern.

## Parsed Expression Integration (B3)

We now evaluate expressions using `ExprEnv` and `eval_expr` over the same X, Y grid and a parameter map, re-using the shared `eval_and_compare` helper.

We:
- Define grids X, Y (see setup cell) and a parameter map.
- Evaluate a list of expression strings.
- Optionally compare against manual constructions and report max absolute difference.
- Include a Gaussian with parameters (a, b, sigma) and a combo expression.
- Print simple timing snapshots.

In [25]:
// Recreate grid (idempotent) and import parser
use std::collections::HashMap;
use candle_notebooks::{Tensor, Device, DType};
use candle_notebooks::expr::{ExprEnv, eval_expr};

let env: ExprEnv = (|| -> candle_notebooks::ah::Result<ExprEnv> {
    let dev = Device::Cpu;
    let n: usize = 128;
    let xs: Vec<f32> = (0..n).map(|i| (i as f32)/(n as f32 - 1.0) * 2.0 - 1.0).collect();
    let ys: Vec<f32> = xs.clone();
    let x: Tensor = Tensor::from_vec(xs.clone(), (n,), &dev)?.to_dtype(DType::F32)?;
    let y: Tensor = Tensor::from_vec(ys.clone(), (n,), &dev)?.to_dtype(DType::F32)?;
    let xx: Tensor = x.unsqueeze(0)?.broadcast_as((n, n))?; // row-wise
    let yy: Tensor = y.unsqueeze(1)?.broadcast_as((n, n))?; // column-wise

    let mut params: HashMap<String, f64> = HashMap::new();
    params.insert("a".to_string(), 1.0f64);
    params.insert("b".to_string(), 1.0f64);
    params.insert("sigma".to_string(), 0.35f64);

    let env = ExprEnv::new(xx.clone().to_dtype(DType::F64)?, yy.clone().to_dtype(DType::F64)?, params)?;
    println!("Grid & env ready: shape={:?}", env.x.shape());
    Ok(env)
})().unwrap();

Grid & env ready: shape=[128, 128]


In [44]:
// eval_and_compare contract
// Inputs: label &str, expr &str (parser syntax), env &ExprEnv (x,y,params),
//         manual: Option<F> builder for a manual Tensor (F: Fn() -> CandleResult<Tensor>).
// Behavior: evaluates expr via eval_expr (f32), optionally computes max_abs_diff vs manual,
//           prints timing and diff. Returns anyhow::Result<()>.
use std::time::Instant;
use candle_notebooks::{Tensor, DType};
use candle_notebooks::expr::{ExprEnv, eval_expr};

pub fn eval_and_compare<F>(
    label: &str, expr: &str, env: &ExprEnv, manual: Option<F>
 ) -> candle_notebooks::ah::Result<()>
where
    F: Fn() -> candle_notebooks::CandleResult<Tensor>,
{
    let t0 = Instant::now();
    let parsed = eval_expr(expr, env)?; // returns f32 tensor
    let dt = t0.elapsed();
    if let Some(build) = manual {
        let manual_t = build()?; // assume already f32 shape match
        let diff = (&parsed - &manual_t)?.abs()?.flatten_all()?.max(0)?;
        println!( 
            "{label}: expr='{}' max_abs_diff={:.3e} time_ms={:.2}",
            expr,
            diff.to_scalar::<f32>()?,
            dt.as_secs_f64() * 1e3
        );
    } else {
        println!("{label}: expr='{}' time_ms={:.2}", expr, dt.as_secs_f64() * 1e3);
    }
    Ok(())
}

In [43]:
// try_cell! macro
// Why: Rust’s `?` operator requires being inside a function/closure returning Result.
// In notebooks, top-level code isn’t a function, so `?` doesn’t compile at top-level.
// This macro wraps a token block in a small closure returning anyhow::Result, then unwraps it,
// letting you write concise cells with `?` as usual without repeating closure boilerplate.
//
// How to use:
//   try_cell!({
//       // your fallible code with `?` goes here
//       some_op()?;
//       Ok(())
//   });
//
// Notes:
// - This unwraps at the end, so an error will abort the cell with a readable anyhow error.
// - Keep Ok(()) at the end of the block so it type-checks.
// - If you prefer no macro at all, swap `?` with `.unwrap()` or define tiny `.u()` helpers.
macro_rules! try_cell {
    ($body:tt) => {{
        (|| -> candle_notebooks::ah::Result<()> $body)().unwrap();
    }};
}

println!("try_cell! macro ready");

try_cell! macro ready


In [45]:
// Baseline evals using try_cell!: compares parser output to manual tensors (radial, checker),
// then runs a pure expr (gaussian_basic). Expects small max_abs_diff (~1e-7).
try_cell!({
    use candle_notebooks::{Tensor, DType};

    // Prepare some manual comparison builders (capturing xx, yy)
    let xx_f32 = env.x.to_dtype(DType::F32)?;
    let yy_f32 = env.y.to_dtype(DType::F32)?;

    // Radial
    eval_and_compare("radial", "sqrt(x*x + y*y)", &env, Some(|| {
        let xx_sq = (&xx_f32 * &xx_f32)?;
        let yy_sq = (&yy_f32 * &yy_f32)?;
        let sum = (xx_sq + yy_sq)?;
        Ok(sum.sqrt()?)
    }))?;

    // Checkerboard-like product of sines
    eval_and_compare("checker", "(sin(10*x)*sin(10*y))", &env, Some(|| {
        let x_10 = (&xx_f32 * 10.0)?;
        let y_10 = (&yy_f32 * 10.0)?;
        Ok((x_10.sin()? * y_10.sin()?)?)
    }))?;

    // Gaussian without manual comparator
    eval_and_compare(
        "gaussian_basic",
        "exp(-(x*x + y*y))",
        &env,
        None::<fn() -> candle_notebooks::CandleResult<Tensor>>,
    )?;

    Ok(())
});

radial: expr='sqrt(x*x + y*y)' max_abs_diff=1.192e-7 time_ms=0.14
checker: expr='(sin(10*x)*sin(10*y))' max_abs_diff=3.874e-7 time_ms=0.45
gaussian_basic: expr='exp(-(x*x + y*y))' time_ms=0.19
checker: expr='(sin(10*x)*sin(10*y))' max_abs_diff=3.874e-7 time_ms=0.45
gaussian_basic: expr='exp(-(x*x + y*y))' time_ms=0.19


In [46]:
// B3 evals: parameterized Gaussian (a,b,sigma) and a combo expression. No manual comparator.
// Prints timings; success means parser evaluation completed.
try_cell!({
    eval_and_compare(
        "gaussian_param",
        "a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b",
        &env,
        None::<fn() -> candle_notebooks::CandleResult<Tensor>>,
    )?;

    eval_and_compare(
        "combo",
        "sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))",
        &env,
        None::<fn() -> candle_notebooks::CandleResult<Tensor>>,
    )?;

    println!("Done B3 expression evaluations.");
    Ok(())
});

gaussian_param: expr='a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b' time_ms=1.15
combo: expr='sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))' time_ms=0.75
Done B3 expression evaluations.
combo: expr='sin(3*x) * cos(5*y) + 0.25 * exp(-(x*x + y*y))' time_ms=0.75
Done B3 expression evaluations.


In [47]:
// Validation: re-run key expressions, asserting parser vs manual agree (tiny max_abs_diff) and
// that pure-expr cases parse and execute. Panics on error via try_cell! unwrap.
try_cell!({
    use candle_notebooks::{Tensor, DType};

    // Radial distance with manual comparator
    eval_and_compare("radial", "sqrt(x*x + y*y)", &env, Some(|| {
        let xx_f32 = env.x.to_dtype(DType::F32)?;
        let yy_f32 = env.y.to_dtype(DType::F32)?;
        let xx_sq = (&xx_f32 * &xx_f32)?;
        let yy_sq = (&yy_f32 * &yy_f32)?;
        let sum = (xx_sq + yy_sq)?;
        Ok(sum.sqrt()?)
    }))?;

    // Checkerboard-style product of sines
    eval_and_compare("checker", "(sin(10*x)*sin(10*y))", &env, Some(|| {
        let xx_f32 = env.x.to_dtype(DType::F32)?;
        let yy_f32 = env.y.to_dtype(DType::F32)?;
        let x_10 = (&xx_f32 * 10.0)?;
        let y_10 = (&yy_f32 * 10.0)?;
        Ok((x_10.sin()? * y_10.sin()?)?)
    }))?;

    // Pure expr-only checks (no manual comparator)
    eval_and_compare(
        "gaussian_basic",
        "exp(-(x*x + y*y))",
        &env,
        None::<fn() -> candle_notebooks::CandleResult<Tensor>>,
    )?;
    eval_and_compare(
        "gaussian_param",
        "a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b",
        &env,
        None::<fn() -> candle_notebooks::CandleResult<Tensor>>,
    )?;

    println!("Validation completed ✅");
    Ok(())
});

radial: expr='sqrt(x*x + y*y)' max_abs_diff=1.192e-7 time_ms=0.17
checker: expr='(sin(10*x)*sin(10*y))' max_abs_diff=3.874e-7 time_ms=0.42
gaussian_basic: expr='exp(-(x*x + y*y))' time_ms=0.25
gaussian_param: expr='a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b' time_ms=0.98
Validation completed ✅
checker: expr='(sin(10*x)*sin(10*y))' max_abs_diff=3.874e-7 time_ms=0.42
gaussian_basic: expr='exp(-(x*x + y*y))' time_ms=0.25
gaussian_param: expr='a * exp(-((x*x)/(2*sigma*sigma) + (y*y)/(2*sigma*sigma))) * b' time_ms=0.98
Validation completed ✅
